# detect

> Read what you paste into the Start box, and say what abhyasa will do with it. No network call.

The Learn page has one box to start something new. You paste a link, a paper, a topic, a file name or some text. As you type,
one quiet line says what abhyasa will do. When two readings are both likely, the box asks one short question. Options show
only when they change the result.

`detect` makes this reading from the text only. It makes no network call and it resolves no host name. The one lookup is a
match against the titles of your vault documents, and the caller does that lookup (`vault_title`).

A goal can name its sources: a folder on this Mac, a GitHub repo, a folder that is in your vault, or vedicreader.
`named_sources` finds them by name. The caller reads the folder names from the disk one time (`folder_names`), and
`detect` only looks them up, so it still makes no network call and reads no disk.

In [ ]:
#| default_exp detect

In [ ]:
#| hide
import os, tempfile
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

Import the link rules of `sources`, so the box and a walk from a link agree on what a paper and a local address are.

In [ ]:
#| export
import re, threading, unicodedata
from pathlib import Path
from urllib.parse import urlparse
from fastcore.utils import *
from abhyasa.sources import ARXIV_ID, ARXIV_LINK, as_link, is_paper, _local, gh_name, folder_why, can_clone, NO_GHEASY
from abhyasa.ground import SKIP_DIRS
from abhyasa.settings import val

## The words

Each reading has one quiet line. The numbers in the lines come from the settings, so a line is never false.

In [ ]:
#| export
HINT = 'Not sure what to paste? Try a topic, such as how attention works.'
LINES = dict(
    arxiv='An arXiv paper. abhyasa reads it in full and makes a walk of up to {steps} ideas.',
    pdf_link='A PDF link. abhyasa reads it in full and makes a walk of up to {steps} ideas.',
    pdf='A PDF. abhyasa reads it in full.',
    repo='A GitHub repository. abhyasa makes a tool walk. Code steps open in Leela.',
    site='A docs site. abhyasa reads the pages in this folder. It reads at most {pages} pages.',
    topic='A topic. The tutor plans a walk and looks for sources on the web.',
    own='A topic. The tutor plans a walk from its own knowledge. Passages are not word for word.',
    text='Pasted text, {lines}. abhyasa finds the ideas in it and makes a walk.',
    file='A text file. abhyasa finds the ideas in it.',
    doc='A document in your vault. abhyasa reads it word for word and makes a walk.',
    video='Video links do not work yet.',
    local='abhyasa does not read links to this computer or a local network.',
    folder='A folder on this Mac. abhyasa files it in your vault.',
    folder_n='A folder with {n}. abhyasa files them in your vault.',
    sources='{n} sources. abhyasa files them in your vault and makes one walk through them.',
    goal='A goal. abhyasa plans a walk that ends at your goal. It reads the sources below first, then looks on the web for what they do not have.',
    goal_own="A goal. abhyasa plans a walk that ends at your goal. It reads the sources below. Ideas with no passage keep the tutor's words.",
    goal_alone='A goal. abhyasa plans a walk that ends at your goal. It looks for sources on the web.',
    goal_alone_own='A goal. abhyasa plans a walk that ends at your goal. The tutor uses its own knowledge.',
    file_name='A file name. To read a file, drop it on this box.')

## Folding a name

A title matches when it is the same text with case and marks folded, so `Maghā` finds `magha`. Only case and marks fold:
`nakṣatras` does not find `Nakshatras`. This is the fold of the concepts list.

In [ ]:
#| export
def fold(s) -> str:
    "`s` in lower case with no marks, and with the spaces at its ends removed."
    return ''.join(ch for ch in unicodedata.normalize('NFKD', str(s or '')) if not unicodedata.combining(ch)).lower().strip()

In [ ]:
test_eq(fold('  Maghā '), 'magha')

## One source

`source_of` reads one word as a source: a paper, a GitHub repository, a docs site, a video, a local link or a folder. A word
with no shape of a source gives None. A bare `github.com/...`, a name with a dot and a slash, or a host with a known ending such as `example.com`, gets
`https://`. The video
hosts are the video hosts of `research.SOCIAL`. A folder is a path that starts with `/` or `~`, or a folder name in `folders`. A bare `owner/name`, such as `fastai/diffusion-nbs`, is a GitHub repository.

In [ ]:
#| export
VIDEO = ('youtube.com', 'youtu.be', 'vimeo.com')
GITHUB = ('github.com', 'www.github.com')
BARE = re.compile(r'[\w-]+(?:\.[\w-]+)+/\S*')   # a link with no scheme, such as github.com/a/b or docs.python.org/3/
TLDS = 'com org net io dev edu gov ai co app info me uk de fr in jp eu ca au us site page'.split()
DOMAIN = re.compile(rf'(?:[\w-]+\.)+(?:{"|".join(TLDS)})', re.I)   # a host with no scheme and no slash, such as example.com; node.js is not one
FILE_NAME = re.compile(r'[^/\\]+\.(?:pdf|md|txt|docx?|epub|ipynb|html?)', re.I)   # a file name, such as paper.pdf

def as_url(s:str) -> str|None:
    "`s` as a web link: as it is, with `https://` for a bare link, or None."
    if re.fullmatch(r'https?://\S+', s, re.I): return s
    return 'https://' + s if BARE.fullmatch(s) or DOMAIN.fullmatch(s) else None

def source_of(s:str, folders=()) -> tuple|None:
    "The kind of the one word `s` as a source, and its link or path; None when `s` does not have the shape of a source."
    s = s.strip()
    if re.match(r'arxiv:', s, re.I) or ARXIV_ID.fullmatch(s) or ARXIV_LINK.fullmatch(s): return 'paper', as_link(s) or s
    if s.startswith(('/', '~')) or (s and fold(s) in {fold(f) for f in folders}): return 'folder', s
    if s.lower().startswith('file:'): return 'local', s
    if not as_url(s) and (g := gh_name(s)): return 'repo', f'https://github.com/{g}'
    if not (u := as_url(s)): return None
    try: host = (urlparse(u).hostname or '').lower()
    except ValueError: return None
    if not host: return None
    if any(host == h or host.endswith('.' + h) for h in VIDEO): return 'video', u
    if _local(host): return 'local', u
    if is_paper(u): return 'paper', u
    if host in GITHUB and len([p for p in urlparse(u).path.split('/') if p]) >= 2: return 'repo', u
    return 'site', u

A word with no shape of a source, such as a topic, gives None. `_local` resolves no name, so `printer.local` is local with no lookup.

In [ ]:
test_eq(source_of('github.com/a/b'), ('repo', 'https://github.com/a/b'))
test_eq(source_of('arxiv:1706.03762'), ('paper', '1706.03762'))
test_eq(source_of('~/code/course22')[0], 'folder')
test_eq(source_of('course22', folders=('course22',))[0], 'folder')
test_eq(source_of('http://printer.local/')[0], 'local')
test_eq(source_of('attention'), None)
test_eq(source_of('example.com'), ('site', 'https://example.com'))
test_eq(source_of('node.js'), None)
test_eq(source_of('fastai/diffusion-nbs'), ('repo', 'https://github.com/fastai/diffusion-nbs'))
test_eq(source_of('and/or'), None)

## Named sources

`folder_names` reads the folder names under the source folders (the setting `sources.roots`), `depth` levels down. It
leaves out dot folders and the build folders of `SKIP_DIRS`. It folds each name, so `Course22` finds `course22`. Two
folders can have the same name, so each name gives a list of paths. This is the one part that reads the disk. The app
does it one time and keeps the result.

In [ ]:
#| export
def folder_names(roots, depth:int=2) -> dict:
    "Each folder name under `roots`, folded, to its paths: `depth` levels down, with no dot folder and no `SKIP_DIRS` folder."
    out = {}
    def walk(p, d):
        try: kids = sorted(x for x in p.iterdir() if not x.name.startswith('.') and x.name not in SKIP_DIRS and x.is_dir() and not x.is_symlink())
        except OSError: return
        for k in kids:
            out.setdefault(fold(k.name), []).append(k)
            if d > 1: walk(k, d - 1)
    for r in map(lambda r: Path(r).expanduser(), roots):
        if r.is_dir(): walk(r, depth)
    return out

A small tree: two folders named `notes` at two depths, a dot folder, and a folder three levels down that is too deep.

In [ ]:
import tempfile

In [ ]:
top = Path(tempfile.mkdtemp())
for d in ('course22/nbs/deep', 'other/Notes', 'notes', '.hidden/x', 'proj/node_modules'): (top/d).mkdir(parents=True)
fn = folder_names([top, top/'not-there'])
test_eq(sorted(fn), ['course22', 'nbs', 'notes', 'other', 'proj'])
test_eq(len(fn['notes']), 2)

`named_sources` finds the sources that a text names. It is pure: it gets the folder names and the roots in the vault, and
makes no disk or network call. Each source is a chip, with:

- `kind`: `folder`, `repo`, `gh`, `doc`, `vedic` or `link`;
- `name`, `path`, `count` (the documents in the vault, when it knows), and `filed` (it is in your vault already);
- `id`: what the Start form sends: `vault:<root>`, `folder:<path>`, `repo:<path>`, `gh:<owner>/<name>`, `doc:<doc_id>`,
  `vedic:ramayana` or `link:<url>`.

A root that is in the vault wins over a folder on the disk with the same name. A name of a common folder, such as `src`
or `data`, is never a source: a goal uses these words for other things. A document title matches only as a whole phrase
of at least 6 characters, with a space or a sign in it, so a one-word title does not match a plain word.

In [ ]:
#| export
COMMON_DIRS = frozenset('src docs doc test tests tools images img data nbs notebooks scripts lib clean slides examples assets static build dist bin config site app apps web api core utils archive backups'.split())
VEDIC = ('vedicreader',)

def _chip(kind, name, path='', count=None, filed=False, id='') -> AttrDict:
    return AttrDict(kind=kind, name=name, path=str(path), count=count, filed=filed, id=id)

def _words(t):
    "The words of `t` that can name a source, with the signs at their ends removed, each one time, in order."
    out = []
    for w in re.findall(r'''[^\s,;()"']+''', t):
        if (w := w.rstrip('.:!?')) and w not in out: out.append(w)
    return out

def named_sources(text:str, titles=(), names:dict=None, roots_in_vault=()) -> list:
    "The sources that `text` names, as chips. `titles`: a dict of title to document id; `names`: from `folder_names`; `roots_in_vault`: the roots, or a dict of root to count."
    t, names, out = (text or '').strip(), names or {}, {}
    rv = roots_in_vault if isinstance(roots_in_vault, dict) else dict.fromkeys(roots_in_vault)
    have = {fold(r): (r, n) for r, n in rv.items()}
    def put(c): out.setdefault(c.id, c)
    def vault(r, n): put(_chip('folder', r, count=n, filed=True, id=f'vault:{r}'))
    for w in ([t] if t.startswith(('/', '~/')) and len(t.splitlines()) == 1 else _words(t)):
        f = fold(w)
        if w.startswith(('/', '~/')):
            p = Path(w).expanduser()   # it reads $HOME, not the disk
            put(_chip('folder', p.name or str(p), p, id=f'folder:{p}'))
        elif f in VEDIC: put(_chip('vedic', 'vedicreader', filed=True, id='vedic:ramayana'))
        elif f in have: vault(*have[f])
        elif g := gh_name(w): vault(*have[fold(g)]) if fold(g) in have else put(_chip('gh', g, f'github.com/{g}', id=f'gh:{g}'))
        elif (s := source_of(w)) and s[0] in ('paper', 'site'): put(_chip('link', w, s[1], id=f'link:{s[1]}'))
        elif f in names and f not in COMMON_DIRS:
            for p in names[f]: put(_chip('folder', Path(p).name, p, id=f'folder:{p}'))
    ft = fold(t)
    for title, did in (titles.items() if isinstance(titles, dict) else ()):
        x = fold(title)
        if len(x) >= 6 and not x.isalnum() and re.search(rf'(?<!\w){re.escape(x)}(?!\w)', ft): put(_chip('doc', title, filed=True, id=f'doc:{did}'))
    return list(out.values())

Two folders with the same name give two chips, each with its path. A common folder name gives none.

In [ ]:
cs = named_sources('Use notes and data, and fastai/fastcore', names=fn)
test_eq([c.id for c in cs], ['folder:' + str(p) for p in fn['notes']] + ['gh:fastai/fastcore'])
test_eq(named_sources('Use fastai/fastcore', roots_in_vault={'fastai/fastcore': 9})[0], _chip('folder', 'fastai/fastcore', count=9, filed=True, id='vault:fastai/fastcore'))
test_eq([c.id for c in named_sources('Read Attention Is All You Need first', {'Attention Is All You Need': 'd1', 'Attention': 'd2'})], ['doc:d1'])

`on_disk` is the disk part, for the caller: a folder chip with a `.git` folder is a local repo (`repo:<path>`), and a
folder that abhyasa does not file gets `why`, from `folder_why`. It does not run `git`: a repo on the disk is text that
abhyasa does not trust. A GitHub chip with no gheasy installed gets `why` too: "Install abhyasa[repos] to add GitHub repos."

In [ ]:
#| export
def on_disk(chips) -> list:
    "`chips`, with what the disk says: a folder with `.git` is a `repo`; a folder that abhyasa does not file, or a GitHub repo with no gheasy to clone it, has `why`."
    for c in chips:
        if c.kind == 'gh' and not can_clone(): c.why = NO_GHEASY   # gheasy, the optional extra that clones, is not installed
        if c.kind != 'folder' or c.filed: continue
        if why := folder_why(c.path): c.why = why
        elif (Path(c.path)/'.git').exists(): c.kind, c.id = 'repo', f'repo:{c.path}'
    return chips

In [ ]:
test_eq(on_disk(named_sources('/tmp/no-such-folder-x'))[0].why, 'abhyasa reads only folders in your home folder.')

## The reading

`detect` gives an `AttrDict`:

- `kind`: one of `empty paper repo site doc topic text ask video local folder sources goal file_name`.
- `line`: the quiet line.
- `ask`: None, or the question, with two answers as (label, kind), and `doc`: the id of the vault document, when it names one.
- `options`: the fields that matter, from `pages tool title subject find`.
- `send`: the fields of `/walk/new` for this reading.
- `sources`: for `sources`, each source as (kind, link or path).
- `chips`: the sources that the text names (`named_sources`). Start files each one that is not in the vault.

A **goal** is one line of 7 to 60 words with "I want", "I would like", "help me", "teach me", "how do I" or "how to", or a
line that names a found source. A **folder** is a path. `counts` holds what the caller found on the disk for a path:
`notebooks`, `files`, `repo`, or `why` it is refused. The folder then asks: "Walk through it" or "Use it for a goal".
`names`, `roots_in_vault` and `docs` (a dict of title to id) go to `named_sources`.

`pick` is the answer to a question: `doc`, `topic` or `text`. `file` is the name of a dropped file: a `.pdf` is in the vault
as the document `doc`, and a `.md` or `.txt` is in the box as text.

In [ ]:
#| export
def _reading(kind, line, ask=None, options=(), send=None, sources=(), chips=()) -> AttrDict:
    return AttrDict(kind=kind, line=line, ask=ask, options=tuple(options), send=dict(send or {}), sources=tuple(sources), chips=list(chips))

def _n(n, w): return f"{n} {w}{'' if n == 1 else 's'}"

def _topic(t, research): return _reading('topic', LINES['topic' if research else 'own'], send=dict(topic=t))

def _text(t, lines): return _reading('text', LINES['text'].format(lines=_n(lines, 'line')), options=('title', 'subject', 'find'), send=dict(text=t))

def _source(kind, u, pages):
    "The reading of one source of `kind` at `u`."
    if kind == 'paper':
        return _reading('paper', LINES['arxiv' if 'arxiv.org/' in u or not u.startswith('http') else 'pdf_link'].format(steps=val('walks.steps')), send=dict(url=u))
    if kind == 'repo': return _reading('repo', LINES['repo'], options=('tool',), send=dict(url=u, tool='on'))
    if kind == 'site': return _reading('site', LINES['site'].format(pages=pages), options=('pages',), send=dict(url=u))
    return _reading(kind, LINES[kind])

GOAL_WORDS = re.compile(r"\b(?:i want|i would like|help me|teach me|how do i|how to)\b")

def _goal(t, chips, research):
    "The reading of a goal, with the sources that it names."
    line = LINES[('goal' if chips else 'goal_alone') + ('' if research else '_own')]
    return _reading('goal', line, send=dict(goal=t, sources=[c.id for c in chips]), chips=chips)

def _folder(p, info, pick):
    "The reading of the folder `p`, with `info` from the disk: refused, or a choice of a walk or a goal."
    p, info = str(Path(p).expanduser()), info if info is not None else {}
    if info.get('why'): return _reading('folder', info['why'])
    k = 'repo' if info.get('repo') else 'folder'
    c = _chip(k, Path(p).name, p, id=f'{k}:{p}')
    n = info.get('notebooks') or 0
    line = LINES['folder_n'].format(n=_n(n, 'notebook') if n else _n(info.get('files') or 0, 'file')) if info else LINES['folder']
    if pick == 'walk': return _reading('folder', line, send=dict(sources=[c.id]), chips=[c])
    return _reading('folder', line, ask=AttrDict(q=line, a=('Walk through it', 'walk'), b=('Use it for a goal', 'goal'), chip=c.id), chips=[c])

In [ ]:
#| export
def detect(text:str, titles=(), research:bool=True, pages:int=8, file:str='', folders=(), pick:str='', doc:str='',
           names:dict=None, roots_in_vault=(), counts:dict=None, docs:dict=None) -> AttrDict:
    "What abhyasa does with `text` from the Start box. `titles`: vault titles to match, or a dict of title to document id; `folders`: folder names that you named."
    t, f = (text or '').strip(), (file or '').lower()
    if t and f.endswith('.pdf'): return _reading('paper', LINES['pdf'], send=dict(ref=doc or t))
    if t and f.endswith(('.md', '.txt')): return _reading('text', LINES['file'], options=('title', 'subject'), send=dict(text=t, title=Path(file).stem))
    if not t: return _reading('empty', HINT)
    lines, words = [l for l in t.splitlines() if l.strip()], t.split()
    parts = [p for p in re.split(r'[\s,]+', t) if p]
    names = names or {}
    found = lambda x: named_sources(x, docs or {}, names, roots_in_vault)
    if len(parts) > 1 and all(srcs := [source_of(p, (*folders, *names)) for p in parts]):
        cs = found(' '.join(parts))
        return _reading('sources', LINES['sources'].format(n=len(srcs)), sources=srcs, send=dict(sources=[c.id for c in cs]), chips=cs)
    if len(lines) == 1 and (s := source_of(t if len(words) == 1 or t.startswith(('/', '~')) else '', (*folders, *names))):
        if s[0] != 'folder': return _source(*s, pages)
        ps = [s[1]] if s[1].startswith(('/', '~')) else names.get(fold(s[1])) or [s[1]]
        return _folder(ps[0], (counts or {}).get(str(Path(ps[0]).expanduser())) if counts is not None else None, pick)
    hit = next((x for x in titles if fold(x) in (fold(t), fold(Path(t).stem))), None) if len(words) <= 6 and len(lines) == 1 else None
    if not hit and len(lines) == 1 and len(words) > 1 and ((cs := found(t)) or (7 <= len(words) <= 60 and GOAL_WORDS.search(fold(t)))): return _goal(t, cs, research)   # a vault title asks first
    if len(lines) > 1 or len(t) > 200: return _text(t, len(lines))
    ref = (titles[hit] if isinstance(titles, dict) else hit) if hit else None
    if FILE_NAME.fullmatch(t):   # a file name is never a topic: the vault document, or a hint to drop the file
        return _reading('doc', LINES['doc'], send=dict(ref=ref)) if hit else _reading('file_name', LINES['file_name'])
    if hit:
        if pick == 'doc': return _reading('doc', LINES['doc'], send=dict(ref=doc or ref))   # the form carries the id of the document
        if pick == 'topic': return _topic(t, research)
        q = f'{hit} is in your vault.'
        return _reading('ask', q, ask=AttrDict(q=q, a=('Walk through that document', 'doc'), b=('Plan a walk on the topic', 'topic'), doc=ref))
    if 7 <= len(words) <= 12 and t.endswith('.'):
        if pick == 'topic': return _topic(t, research)
        if pick == 'text': return _text(t, 1)
        q = 'Is this a topic or a text?'
        return _reading('ask', q, ask=AttrDict(q=q, a=('A topic', 'topic'), b=('A text', 'text')))
    return _text(t, 1) if t.endswith('.') and len(words) > 12 else _topic(t, research)

## The cases

`DETECT_CASES` holds each shape that the box knows, with the kind it must give. The tests of the app use the same table.

In [ ]:
#| export
DETECT_CASES = [('1706.03762', (), 'paper'), ('arxiv:1706.03762v5', (), 'paper'), ('  https://arxiv.org/abs/1706.03762  ', (), 'paper'),
                ('https://x.org/a.pdf', (), 'paper'), ('github.com/vedicreader/fossick', (), 'repo'), ('https://github.com/a/b', (), 'repo'),
                ('https://docs.python.org/3/library/asyncio.html', (), 'site'), ('Nakshatras', ('Nakshatras',), 'ask'),
                ('how attention works', (), 'topic'), ('nakṣatra', (), 'topic'), ('a\nb', (), 'text'), ('x' * 300, (), 'text'),
                ('The moon spends about one day in each of the lunar mansions.', (), 'ask'), ('https://youtu.be/x', (), 'video'),
                ('http://127.1/', (), 'local'), ('http://printer.local/', (), 'local'), ('', (), 'empty'), ('   ', (), 'empty'),
                ('~/code/course22', (), 'folder'), ('/Users/me/My notes', (), 'folder'),
                ('github.com/a/b ~/code/fastbook', (), 'sources'), ('1706.03762\n2005.14165', (), 'sources'),
                ('example.com', (), 'site'), ('node.js', (), 'topic'), ('paper.pdf', (), 'file_name'), ('paper.pdf', ('paper',), 'doc')]
GOAL = 'I want to fine-tune (SFT) my own diffusion model to make my own Ramayana comic from the episodes in vedicreader. Use course22, course22p2 and fastbook, top down. Also fastai/diffusion-nbs.'
DETECT_CASES += [('~/code/x', (), 'folder'), ('/tmp/x', (), 'folder'), ('fastai/diffusion-nbs', (), 'repo'), (GOAL, (), 'goal'),
                 ('help me learn how attention works in a transformer', (), 'goal')]

Each case gives its kind.

In [ ]:
for t, ti, k in DETECT_CASES: test_eq(detect(t, ti).kind, k)

A vault title asks first. Its first answer walks through the document, and the answer `doc` sends its title as `ref`.

In [ ]:
r = detect('Nakshatras', ('Nakshatras',))
test_eq(r.ask.a, ('Walk through that document', 'doc'))
test_eq(r.ask.q, 'Nakshatras is in your vault.')
test_eq(detect('nakshatras', ('Nakshatras',), pick='doc').send, dict(ref='Nakshatras'))
test_eq(detect('Nakshatras', ('Nakshatras',), pick='topic').kind, 'topic')
test_eq(detect('nakshatras', {'Nakshatras': 'd1'}).ask.doc, 'd1')   # with a dict, the question carries the document id
test_eq(detect('nakshatras', {'Nakshatras': 'd1'}, pick='doc', doc='d1').send, dict(ref='d1'))

With research off, the topic line says that the tutor uses its own knowledge.

In [ ]:
assert 'own knowledge' in detect('how attention works', research=False).line
test_eq(detect('how attention works').send, dict(topic='how attention works'))

Pasted text shows Title, Subject and Find concepts only. A sentence of 7 to 12 words answered as a text is one line of text.

In [ ]:
test_eq(detect('a\nb').options, ('title', 'subject', 'find'))
test_eq(detect('a\nb').line, 'Pasted text, 2 lines. abhyasa finds the ideas in it and makes a walk.')
test_eq(detect('The moon spends about one day in each of the lunar mansions.', pick='text').kind, 'text')

A repo ticks the tool box. A site says how many pages it reads. A paper says how many ideas a walk has at most.

In [ ]:
r = detect('github.com/vedicreader/fossick')
test_eq((r.options, r.send), (('tool',), dict(url='https://github.com/vedicreader/fossick', tool='on')))
test_eq(detect('https://docs.python.org/3/', pages=5).line, 'A docs site. abhyasa reads the pages in this folder. It reads at most 5 pages.')
test_eq(detect('1706.03762').line, 'An arXiv paper. abhyasa reads it in full and makes a walk of up to 12 ideas.')
test_eq(detect('arxiv.org/abs/1706.03762').send, dict(url='https://arxiv.org/abs/1706.03762'))

A file name is never a topic: it is the vault document of that name, or a hint to drop the file. A dropped file is certain: the box never asks for it.

In [ ]:
test_eq(detect('Nakshatras', ('Nakshatras',), file='naks.pdf').line, 'A PDF. abhyasa reads it in full.')
test_eq(detect('My paper', file='My paper.pdf', doc='d7').send, dict(ref='d7'))
r = detect('Line one.\nLine two.', file='notes.md')
test_eq((r.kind, r.line, r.options, r.send['title']), ('text', 'A text file. abhyasa finds the ideas in it.', ('title', 'subject'), 'notes'))

Several sources at once give each source. A folder name that you named is a folder.

In [ ]:
r = detect('github.com/a/b, ~/code/fastbook')
test_eq(r.sources, (('repo', 'https://github.com/a/b'), ('folder', '~/code/fastbook')))
test_eq(r.send['sources'], ['gh:a/b', 'folder:' + str(Path('~/code/fastbook').expanduser())])
test_eq(detect('course22', folders=('course22',)).kind, 'folder')

A goal names its sources. Here the folder names come from a fake disk, and `course22p2` is in the vault. The chips are a
folder on the disk, a folder in the vault, a GitHub repo and vedicreader.

In [ ]:
names = {'course22': [Path('~/code/personal/orgs/course22').expanduser()], 'fastbook': [Path('/h/o/fastbook')]}
r = detect(GOAL, names=names, roots_in_vault=['course22p2'])
test_eq(sorted(c.id for c in r.chips), ['folder:' + str(names['course22'][0]), 'folder:/h/o/fastbook', 'gh:fastai/diffusion-nbs', 'vault:course22p2', 'vedic:ramayana'])
assert 'reads the sources below first' in r.line
test_eq(r.send['goal'], GOAL)

With research off, the goal line says that ideas with no passage keep the tutor's words. A goal that names no source says that abhyasa looks on the web.

In [ ]:
assert "keep the tutor's words" in detect(GOAL, research=False).line
test_eq(detect('help me learn how attention works in a transformer').line, LINES['goal_alone'])

A folder with its count from the disk asks what to do. `pick='walk'` sends the folder. A refused folder says why.

In [ ]:
p = str(Path('~/code/x').expanduser())
r = detect('~/code/x', counts={p: dict(notebooks=44, files=50)})
test_eq((r.line, r.ask.a, r.ask.b, r.ask.chip), ('A folder with 44 notebooks. abhyasa files them in your vault.', ('Walk through it', 'walk'), ('Use it for a goal', 'goal'), f'folder:{p}'))
test_eq(detect('~/code/x', counts={p: dict(notebooks=0, files=1, repo=True)}, pick='walk').send, dict(sources=[f'repo:{p}']))
test_eq(detect('/tmp/x', counts={'/tmp/x': dict(why='abhyasa reads only folders in your home folder.')}).line, 'abhyasa reads only folders in your home folder.')
test_eq(detect('course22', names=names).chips[0].path, str(names['course22'][0]))

## The vault title

`vault_title` finds the one document on the main shelf whose title is the text, with case and marks folded, and gives its title
and its id. It skips your walks, talks and concepts. A text of more than 6 words never matches. The box asks after each pause in
your typing, so the folded titles are kept in an index. One small query checks if the vault changed (its count of documents, the
time of the newest, and the length of all titles); only then is the index made again.

In [ ]:
#| export
SKIP_KINDS = ('journey', 'talk', 'concept')
_INDEX, _ILOCK = {}, threading.Lock()   # the path of a vault -> (its version, the folded titles -> (title, id))

def _where(): return f"(kind is null or kind not in ({','.join('?' * len(SKIP_KINDS))}))", list(SKIP_KINDS)

def title_index(vault) -> dict:
    "The folded titles of the documents on `vault`, each to (title, id). Made again only when the vault changed."
    w, a = _where()
    t = vault.t.docs.name
    v = tuple(first(vault.db.execute(f'select count(*), max(added_at), total(length(title)) from [{t}] where {w}', a).fetchall()))
    key = (str(getattr(vault, 'path', id(vault))), t)
    with _ILOCK:
        if (got := _INDEX.get(key)) and got[0] == v: return got[1]
    idx = {}
    for r in vault.t.docs(select='id, title', where=w, where_args=a): idx.setdefault(fold(r['title']), (r['title'], r['id']))
    with _ILOCK: _INDEX[key] = (v, idx)
    return idx

def vault_title(vault, q:str) -> tuple|None:
    "The (title, id) of the document on `vault` (the main shelf) that is `q`, or `q` with no file ending, folded; None when there is none."
    f = fold(q)
    if not f or len(f.split()) > 6: return None
    idx = title_index(vault)
    return idx.get(f) or idx.get(fold(Path(q).stem))

A test vault with one document named Nakshatras. A part of the title does not match. A new document shows at once.

In [ ]:
from abhyasa.store import temp_learning

In [ ]:
lr = temp_learning()
d = lr.vault.add('Some text about nakshatras.', title='Nakshatras', source='test://n')
test_eq(vault_title(lr.vault, ' NAKSHATRAS '), ('Nakshatras', d['doc_id']))
test_eq(vault_title(lr.vault, 'naksha'), None)
lr.vault.add('Some text about rasis.', title='Rasis', source='test://r')
test_eq(vault_title(lr.vault, 'rasis.pdf')[0], 'Rasis')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()